# Getting Started: 2D Conservative Transport on the Willowbend Creek Mesh

This notebook is a first-look tour of the ClearWater-Riverine transport solver. It loads the Willowbend Creek HEC-RAS 2D plan, configures a single conservative tracer, runs the solver on the computational mesh, and produces velocity and concentration plots.

Willowbend Creek is a 2 km by 2 km synthetic lowland stream with a meander bend and an oxbow backwater. Three boundaries drive the domain: a steady upstream inflow, a cool spring-fed tributary at the inside of the meander, and a powerplant discharge into the oxbow. Inputs and a precomputed simulation output are bundled under `docs/examples/data/willowbend_creek/`.


## 1. Imports and paths

Run this notebook from the documentation repository root. The relative path `docs/examples/data/willowbend_creek/` must resolve to the bundled test-case directory.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.colors import Normalize

import clearwater_riverine as cwr
from clearwater_riverine.config import init_from_file

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.size": 10,
})


In [ ]:
data_dir = Path.cwd() / "docs" / "examples" / "data" / "willowbend_creek"
config_path = data_dir / "riverine.yml"

assert (data_dir / "clearWaterTestCases.p48.hdf").exists(), data_dir
assert config_path.exists(), config_path


## 2. Load the transport configuration and run the model

`riverine.yml` declares the simulation directory, the HEC-RAS 2D plan HDF, the simulation start and end times, the time step, the chunk size, the diffusion coefficient, and the constituents to transport. The bundled configuration carries a single `tracer` constituent with constant initial and boundary values, which keeps the demonstration focused on the transport engine itself.


In [ ]:
model = init_from_file(config_path)
model


In [ ]:
%time model.run()


## 3. Open the simulation output

`model.run()` writes the simulated state to a `zarr` store referenced from `riverine.yml`. The bundled `model_outputs.zarr` is the same store written by a previous run and can be opened directly with `xarray`.


In [ ]:
results = xr.open_zarr(data_dir / "model_outputs.zarr")
results


## 4. Plot the velocity field on the computational mesh

The HEC-RAS 2D solution provides a face-averaged flow on every internal edge of the unstructured mesh. The cell-averaged velocity is reconstructed from the face flows and plotted below as a quiver overlay on the mesh.


In [ ]:
node_x = results["node_x"].values
node_y = results["node_y"].values
face_nodes = results["face_nodes"].values  # (nface, nmax_face), -1 fill

u = results["velocity_x"].isel(time=-1).values
v = results["velocity_y"].isel(time=-1).values

face_x = []
face_y = []
for fn in face_nodes:
    valid = fn[fn >= 0]
    face_x.append(node_x[valid].mean())
    face_y.append(node_y[valid].mean())
face_x = np.array(face_x)
face_y = np.array(face_y)

fig, ax = plt.subplots(figsize=(8, 7))
polys = []
for fn in face_nodes:
    valid = fn[fn >= 0]
    polys.append(MplPolygon(np.column_stack([node_x[valid], node_y[valid]])))
pc = PatchCollection(polys, facecolor="#f0f0f0", edgecolor="#bdbdbd", linewidth=0.2)
ax.add_collection(pc)
ax.quiver(face_x, face_y, u, v, scale=20, width=0.0025, color="#1f77b4")
ax.set_aspect("equal")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Willowbend Creek velocity field at end of simulation")
fig.tight_layout()


## 5. Plot tracer concentration at representative cells

The bundled `tracer` constituent is initialized at 100 throughout the domain and held at 100 at every boundary, so a conservative-transport solution should remain near 100 everywhere. Departures from 100 indicate numerical diffusion or boundary-flux artifacts and are useful for sanity-checking a new mesh or a new boundary configuration.


In [ ]:
representative_cells = {
    "upstream": 250,
    "meander": 1250,
    "oxbow": 2400,
}

fig, ax = plt.subplots(figsize=(9, 4))
for label, cell in representative_cells.items():
    series = results["tracer"].isel(nface=cell).to_pandas()
    ax.plot(series.index, series.values, label=label)
ax.set_xlabel("Time")
ax.set_ylabel("Tracer concentration")
ax.set_title("Tracer concentration at three representative cells")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()


## 6. Plot the tracer field at end of simulation


In [ ]:
tracer_end = results["tracer"].isel(time=-1).values

fig, ax = plt.subplots(figsize=(8, 7))
polys = []
for fn in face_nodes:
    valid = fn[fn >= 0]
    polys.append(MplPolygon(np.column_stack([node_x[valid], node_y[valid]])))
pc = PatchCollection(polys, edgecolor="#666666", linewidth=0.1)
pc.set_array(tracer_end)
pc.set_norm(Normalize(vmin=99.0, vmax=101.0))
pc.set_cmap("viridis")
ax.add_collection(pc)
ax.autoscale()
ax.set_aspect("equal")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Tracer concentration at end of simulation")
cbar = fig.colorbar(pc, ax=ax, shrink=0.8)
cbar.set_label("Tracer")
fig.tight_layout()


## Next steps

- Couple the transport solver to the Temperature Simulation Module (TSM): see [Willowbend Creek Temperature](./02_willowbend_temperature.ipynb).
- Add the Nutrient Simulation Module (NSM1) to a coupled run: see [Willowbend Creek Nutrients](./03_willowbend_nutrients.ipynb).
- Apply the transport engine to a real-world domain: see [Ohio River Transport](./04_ohio_river_transport.ipynb).
